In [13]:
import pandas as pd
import numpy as np 


In [63]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score


In [64]:
### Load Data


In [65]:
df = pd.read_csv('Customer-Churn.csv')

In [66]:
### Basic Cleaning

In [67]:
### Remove useless column 

In [68]:
df = df.drop("customerID", axis=1)

In [69]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce'
)

df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"]).median()
df["Churn"]


0        No
1        No
2       Yes
3        No
4       Yes
       ... 
7038     No
7039     No
7040     No
7041    Yes
7042     No
Name: Churn, Length: 7043, dtype: object

In [70]:

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0
})
df["Churn"]

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [71]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [72]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

In [73]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

In [74]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,     stratify=y, random_state=42)

In [75]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

In [76]:
selector_model = RandomForestClassifier(n_estimators=300, random_state=42)


In [77]:
selector_model.fit(X_train_prep, y_train)

selector = SelectFromModel(
  selector_model,
  threshold="median", 
  prefit=True)

X_train_sel = selector.transform(X_train_prep)
X_test_sel = selector.transform(X_test_prep)  

print("Original features:", X_train_prep.shape[1])
print("Selected features:", X_train_sel.shape[1])

Original features: 45
Selected features: 23


In [78]:
model = RandomForestClassifier(
  n_estimators=300,
  max_depth=10,
  class_weight="balanced",
  random_state=42
  )

model.fit(X_train_sel, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=300,
                       random_state=42)

In [79]:
pred = model.predict(X_test_sel)
prob = model.predict_proba(X_test_sel)[:, 1]

In [80]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred)) 

print("\nClassification Report:")
print(classification_report(y_test, pred))  

print("\nROC AUC:", roc_auc_score(y_test, prob))


Confusion Matrix:
[[820 215]
 [113 261]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.79      0.83      1035
           1       0.55      0.70      0.61       374

    accuracy                           0.77      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.77      0.78      1409


ROC AUC: 0.8335735875377819


In [89]:
ohe = preprocessor.named_transformers_["cat"].named_steps["encoder"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)


In [82]:
all_feature_names = list(num_cols) + list(cat_feature_names)

In [83]:
selected_mask = selector.get_support()

In [84]:
selected_features = np.array(all_feature_names)[selected_mask]

In [85]:
print("\nSelected Features Names:")
for f in selected_features:
    print(f)  


Selected Features Names:
SeniorCitizen
tenure
MonthlyCharges
gender_Female
gender_Male
Partner_No
Partner_Yes
MultipleLines_No
MultipleLines_Yes
InternetService_Fiber optic
OnlineSecurity_No
OnlineBackup_No
OnlineBackup_Yes
DeviceProtection_No
DeviceProtection_Yes
TechSupport_No
Contract_Month-to-month
Contract_Two year
PaperlessBilling_No
PaperlessBilling_Yes
PaymentMethod_Bank transfer (automatic)
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
